In [ ]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

DATA_ROOT = Path("./IDEABench")
CROMA_PY_DIR = Path(".")
PRETRAINED_PATH = Path("./checkpoints/croma_base.pt")
OUTPUT_DIR = Path("./runs/ideabench_croma_base")

EPOCHS = 30
BATCH_SIZE = 2
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
NUM_WORKERS = 0 
SEED = 42
LABEL_OFFSET = 0
IGNORE_INDEX = 255 
CLASS_WEIGHTS = (1.0, 1.0, 1.0)
TRAIN_CITIES = None 

S2_INPUT_BANDS = (
    "B02", "B03", "B04", "B05", "B06",
    "B07", "B08", "B8A", "B11", "B12",
)
CROMA_BANDS = (
    "B01", "B02", "B03", "B04", "B05", "B06",
    "B07", "B08", "B8A", "B09", "B11", "B12",
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)
print("Data root:", DATA_ROOT.resolve())

In [ ]:
assert DATA_ROOT.is_dir(), f"Dataset directory not found: {DATA_ROOT}"
assert PRETRAINED_PATH.is_file(), f"Checkpoint not found: {PRETRAINED_PATH}"
assert (CROMA_PY_DIR / "use_croma.py").is_file(), "Place use_croma.py in CROMA_PY_DIR."

city_dirs = sorted(path for path in DATA_ROOT.iterdir() if path.is_dir())
print("Available cities:", [path.name for path in city_dirs])

EXAMPLE_CITY_DIR = next(
    (path for path in city_dirs if "buenos" in path.name.lower()),
    city_dirs[0],
)
EXAMPLE_SPLIT = "train"
EXAMPLE_ID = "67"
example_folder = EXAMPLE_CITY_DIR / EXAMPLE_SPLIT
example_paths = {
    prefix: example_folder / f"{prefix}_{EXAMPLE_ID}.tif"
    for prefix in ("S1", "S2", "RF")
}
if not all(path.exists() for path in example_paths.values()):
    reference = next(iter(sorted(example_folder.glob("RF_*.tif"))), None)
    if reference is None:
        raise FileNotFoundError(f"No RF masks found in {example_folder}")
    EXAMPLE_ID = reference.stem.split("RF_", 1)[1]
    example_paths = {
        prefix: example_folder / f"{prefix}_{EXAMPLE_ID}.tif"
        for prefix in ("S1", "S2", "RF")
    }

for prefix, path in example_paths.items():
    with rasterio.open(path) as source:
        array = source.read()
        print(f"\n{prefix}: {path}")
        print("Shape:", array.shape, "dtype:", array.dtype)
        print("Descriptions:", source.descriptions, "NoData:", source.nodata)
        for index, band in enumerate(array, start=1):
            finite = band[np.isfinite(band)]
            print(
                f"  Band {index}: min={finite.min():.3f}, "
                f"max={finite.max():.3f}, mean={finite.mean():.3f}"
            )
        if prefix == "RF":
            print("RF unique values:", np.unique(array, return_counts=True))

In [ ]:
## Dataset 

def normalize_channels(image):
    """Normalize each channel while safely handling non-finite values."""
    image = image.astype(np.float32, copy=False)
    normalized = np.zeros_like(image, dtype=np.float32)

    for channel_index in range(image.shape[0]):
        channel = image[channel_index]
        valid_mask = np.isfinite(channel)

        if not valid_mask.any():
            continue

        valid_values = channel[valid_mask]

        mean = valid_values.mean()
        std = valid_values.std()

        if std < 1e-6:
            continue

        lower = mean - 2.0 * std
        upper = mean + 2.0 * std

        normalized[channel_index, valid_mask] = np.clip(
            (valid_values - lower) / (upper - lower),
            0.0,
            1.0,
        )

    return normalized

def read_tif(path, expected_channels):
    """Read a pre-aligned 128-by-128 raster without resampling."""
    with rasterio.open(path) as source:
        if source.count != expected_channels or source.shape != (128, 128):
            raise ValueError(f"Unexpected raster dimensions: {path}")
        image = source.read()
        if source.nodata is not None and np.any(image == source.nodata):
            if expected_channels != 1:
                raise ValueError(f"Sensor NoData found in {path}")
        return image


def expand_s2(image):
    """Insert absent optical bands without changing CROMA weights."""
    expanded = np.zeros((12, 128, 128), dtype=np.float32)
    for source_index, band_name in enumerate(S2_INPUT_BANDS):
        expanded[CROMA_BANDS.index(band_name)] = image[source_index]
    return expanded


class IDEABenchDataset(Dataset):
    """Load S1, S2 and RF patches matched by filename identifier."""

    def __init__(self, root, split, cities=None):
        self.samples = []
        root = Path(root)
        folders = sorted(root.glob(f"*/{split}"))
        if cities is not None:
            folders = [p for p in folders if p.parent.name in cities]
        for folder in folders:
            labels = sorted(folder.glob("RF_*.tif"))
            for label_path in labels:
                suffix = label_path.name.removeprefix("RF_")
                sar_path = folder / f"S1_{suffix}"
                optical_path = folder / f"S2_{suffix}"
                if not sar_path.exists() or not optical_path.exists():
                    raise FileNotFoundError(f"Missing S1/S2 for {label_path}")
                self.samples.append((sar_path, optical_path, label_path))
        if not self.samples:
            raise ValueError(f"No samples found for split={split}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sar_path, optical_path, label_path = self.samples[index]
        sar = normalize_channels(read_tif(sar_path, 2))
        optical = expand_s2(normalize_channels(read_tif(optical_path, 10)))
        label = read_tif(label_path, 1)[0]
        if not np.isfinite(label).all():
            raise ValueError(f"Non-finite labels in {label_path}")
        valid = label != IGNORE_INDEX
        if not np.all(label[valid] == np.floor(label[valid])):
            raise ValueError(f"Non-integer labels in {label_path}")
        label = label.astype(np.int64)
        label[valid] -= LABEL_OFFSET
        if not np.isin(label[valid], (0, 1, 2)).all():
            raise ValueError(
                f"Unexpected classes in {label_path}: {np.unique(label)}"
            )
        return (
            torch.from_numpy(sar.copy()),
            torch.from_numpy(optical.copy()),
            torch.from_numpy(label.copy()),
        )


train_dataset = IDEABenchDataset(DATA_ROOT, "train", TRAIN_CITIES)
val_dataset = IDEABenchDataset(DATA_ROOT, "val", TRAIN_CITIES)
print("Training patches:", len(train_dataset))
print("Validation patches:", len(val_dataset))

sar, optical, labels = train_dataset[0]
print("S1:", sar.shape, "S2 padded:", optical.shape, "RF:", labels.shape)
print("Labels:", torch.unique(labels).tolist())
assert sar.shape == (2, 128, 128)
assert optical.shape == (12, 128, 128)
assert labels.shape == (128, 128)
assert torch.all(optical[0] == 0) and torch.all(optical[9] == 0)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
)

In [ ]:
## Model

sys.path.insert(0, str(CROMA_PY_DIR.resolve()))
from use_croma import PretrainedCROMA # from https://github.com/antofuller/CROMA


class CROMASegmenter(nn.Module):
    """Full-fine-tuning model using the original CROMA-base encoder."""

    def __init__(self, pretrained_path):
        super().__init__()
        self.backbone = PretrainedCROMA(
            pretrained_path=str(pretrained_path),
            size="base",
            modality="both",
            image_resolution=128,
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(768 * 3, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.GELU(),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(128, 3, kernel_size=1),
        )

    def forward(self, sar, optical):
        outputs = self.backbone(SAR_images=sar, optical_images=optical)
        spatial_maps = []
        for key in ("SAR_encodings", "optical_encodings", "joint_encodings"):
            tokens = outputs[key]
            spatial_maps.append(
                tokens.transpose(1, 2).reshape(sar.shape[0], 768, 16, 16)
            )
        logits = self.decoder(torch.cat(spatial_maps, dim=1))
        return F.interpolate(
            logits, size=(128, 128), mode="bilinear", align_corners=False
        )


model = CROMASegmenter(PRETRAINED_PATH).to(DEVICE)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
assert all(parameter.requires_grad for parameter in model.backbone.parameters())

with torch.no_grad():
    model.eval()
    sample_logits = model(sar[None].to(DEVICE), optical[None].to(DEVICE))
print("Output:", sample_logits.shape)
assert sample_logits.shape == (1, 3, 128, 128)

In [ ]:
## Loss and metrics

class_weights = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32, device=DEVICE)


def segmentation_loss(logits, target):
    """Compute weighted cross-entropy plus masked soft Dice."""
    valid = target != IGNORE_INDEX
    if not valid.any():
        return logits.sum() * 0.0
    ce = F.cross_entropy(
        logits, target, weight=class_weights, ignore_index=IGNORE_INDEX
    )
    safe_target = target.masked_fill(~valid, 0)
    one_hot = F.one_hot(safe_target, num_classes=3)
    one_hot = one_hot.permute(0, 3, 1, 2).float()
    mask = valid.unsqueeze(1)
    probabilities = logits.softmax(dim=1) * mask
    one_hot = one_hot * mask
    intersection = (probabilities * one_hot).sum(dim=(0, 2, 3))
    denominator = (probabilities + one_hot).sum(dim=(0, 2, 3))
    dice = 1.0 - ((2.0 * intersection + 1e-6) / (denominator + 1e-6)).mean()
    return ce + dice


def metrics_from_confusion(confusion):
    """Calculate per-class IoU, F1 and macro IoU."""
    matrix = confusion.float()
    true_positive = matrix.diag()
    false_positive = matrix.sum(dim=0) - true_positive
    false_negative = matrix.sum(dim=1) - true_positive
    union = true_positive + false_positive + false_negative
    iou = torch.where(union > 0, true_positive / union.clamp(min=1), float("nan"))
    f1_denom = 2 * true_positive + false_positive + false_negative
    f1 = torch.where(
        f1_denom > 0,
        2 * true_positive / f1_denom.clamp(min=1),
        float("nan"),
    )
    return {
        "miou": torch.nanmean(iou).item(),
        "iou": iou.tolist(),
        "f1": f1.tolist(),
        "support": matrix.sum(dim=1).int().tolist(),
    }


def run_epoch(model, loader, optimizer=None):
    """Train or evaluate one epoch and accumulate confusion counts."""
    training = optimizer is not None
    model.train(training)
    confusion = torch.zeros((3, 3), dtype=torch.long, device=DEVICE)
    total_loss = 0.0
    total_images = 0
    for sar_batch, optical_batch, target in tqdm(loader, leave=False):
        sar_batch = sar_batch.to(DEVICE, non_blocking=True)
        optical_batch = optical_batch.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(training):
            logits = model(sar_batch, optical_batch)
            loss = segmentation_loss(logits, target)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
        predicted = logits.argmax(dim=1)
        valid = target != IGNORE_INDEX
        counts = torch.bincount(
            (target[valid] * 3 + predicted[valid]).reshape(-1), minlength=9
        )
        confusion += counts.reshape(3, 3)
        total_loss += loss.item() * target.shape[0]
        total_images += target.shape[0]
    result = metrics_from_confusion(confusion.cpu())
    result["loss"] = total_loss / max(total_images, 1)
    return result

In [ ]:
## Full fine-tuning and best validation checkpoint

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
history = []
best_miou = -1.0
best_path = OUTPUT_DIR / "best_croma_base_segmentation.pt"

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, optimizer=optimizer)
    val_metrics = run_epoch(model, val_loader)
    history.append({"epoch": epoch, "train": train_metrics, "val": val_metrics})
    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"train loss={train_metrics['loss']:.4f}, "
        f"mIoU={train_metrics['miou']:.4f} | "
        f"val loss={val_metrics['loss']:.4f}, "
        f"mIoU={val_metrics['miou']:.4f}"
    )
    if val_metrics["miou"] > best_miou:
        best_miou = val_metrics["miou"]
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "val_metrics": val_metrics,
                "s2_input_bands": S2_INPUT_BANDS,
                "label_offset": LABEL_OFFSET,
                "ignore_index": IGNORE_INDEX,
            },
            best_path,
        )
        print("Saved best model:", best_path)

In [ ]:
## Learning curves

epochs = [entry["epoch"] for entry in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for split in ("train", "val"):
    axes[0].plot(epochs, [entry[split]["loss"] for entry in history], label=split)
    axes[1].plot(epochs, [entry[split]["miou"] for entry in history], label=split)
axes[0].set_title("Loss")
axes[1].set_title("mIoU")
for axis in axes:
    axis.set_xlabel("Epoch")
    axis.legend()
plt.tight_layout()
plt.show()

In [ ]:
## Evaluate on the held-out test blocks and per city

checkpoint = torch.load(best_path, map_location="cpu", weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

for city in city_dirs:
    if TRAIN_CITIES is not None and city.name not in TRAIN_CITIES:
        continue
    try:
        city_test = IDEABenchDataset(DATA_ROOT, "test", [city.name])
    except ValueError:
        continue
    city_loader = DataLoader(
        city_test, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS,
    )
    results = run_epoch(model, city_loader)
    print(city.name, results)

In [ ]:
## Visualize predictions

test_dataset = IDEABenchDataset(DATA_ROOT, "test", TRAIN_CITIES)
sar, optical, target = test_dataset[0]
with torch.no_grad():
    logits = model(sar[None].to(DEVICE), optical[None].to(DEVICE))
prediction = logits.argmax(dim=1)[0].cpu().numpy()

rgb = optical[[3, 2, 1]].permute(1, 2, 0).numpy()
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(rgb)
axes[0].set_title("Sentinel-2 RGB")
axes[1].imshow(target.numpy(), vmin=0, vmax=2, cmap="viridis")
axes[1].set_title("RF ground truth")
axes[2].imshow(prediction, vmin=0, vmax=2, cmap="viridis")
axes[2].set_title("CROMA prediction")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()